<a href="https://colab.research.google.com/github/Ena-AlexBrush/Fine-Tuning-Experiments/blob/main/GRPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [2]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 43.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
!pip install trl[GRPOTrainer]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [4]:
# !pip install trl[vllm]

In [5]:
# !pip install trl[GRPOTrainer]

In [6]:
get_ipython().system('pip uninstall vllm -y') # Uninstall vllm to resolve CUDA version conflict


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [8]:
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig
from trl.rewards import accuracy_reward
import re

In [9]:
# 1. Dataset Preparation
train_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-magpie-ultra", split="train[:20]")
eval_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-magpie-ultra", split="test[:20]")
new_train_dataset = train_dataset.rename_column("messages","prompt")

README.md:   0%|          | 0.00/9.72k [00:00<?, ?B/s]

data/smol-magpie-ultra/train-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  232MB            

data/smol-magpie-ultra/train-00000-of-00(…): downloading bytes:           |  0.00B            

data/smol-magpie-ultra/train-00001-of-00(…): reconstructing file:   0%|          |  0.00B /  233MB            

data/smol-magpie-ultra/train-00001-of-00(…): downloading bytes:           |  0.00B            

...

data/smol-magpie-ultra/train-00005-of-00(…): downloading bytes:           |  0.00B            

data/smol-magpie-ultra/test-00000-of-000(…): reconstructing file:   0%|          |  0.00B / 73.2MB            

data/smol-magpie-ultra/test-00000-of-000(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/409537 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/21555 [00:00<?, ? examples/s]

In [10]:
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M", device_map="auto")

# Ensure tokenizer.chat_template is set
if tokenizer.chat_template is None:
    print("Tokenizer chat_template is not set. Setting a default ChatML-like template.")
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{% if message['role'] == 'user' %}"
        "{{ '<|im_start|>user\n' + message['content'] + '<|im_end|>\n' }}"
        "{% elif message['role'] == 'assistant' %}"
        "{{ '<|im_start|>assistant\n' + message['content'] + '<|im_end|>\n' }}"
        "{% endif %}"
        "{% endfor %}"
        "{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
    )

# Ensure pad_token is set if it's None, often to eos_token for causal LMs
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Tokenizer chat_template is not set. Setting a default ChatML-like template.


In [11]:
print("Chat template support:", hasattr(tokenizer, "apply_chat_template"))

Chat template support: True


In [14]:
print(new_train_dataset.column_names)

# Create prompt_to_gold_completion_map
# This map will store formatted string prompts mapping to string gold completions.

prompt_to_gold_completion_map = {}

for item in train_dataset:
    messages = item['messages'] # Access original 'messages' from train_dataset

    # We need to find the "prompt" part that the model will generate from,
    # and the "gold completion" that it should generate.
    # For a multi-turn conversation, this usually means splitting
    # at the last assistant turn.

    gold_completion = None
    prompt_messages = []

    last_assistant_idx = -1
    for i in range(len(messages) - 1, -1, -1):
        if messages[i]['role'] == 'assistant':
            last_assistant_idx = i
            break

    if last_assistant_idx != -1:
        # The prompt part of the conversation leading up to the gold completion
        prompt_messages = messages[:last_assistant_idx]

        # The gold completion part is the content of the last assistant message
        gold_completion = messages[last_assistant_idx]['content']

        if prompt_messages:
            # Format the prompt messages into a single string using the tokenizer's chat template.
            # This formatted string will be the key in our map.
            # add_generation_prompt=True is crucial here, as it signifies the model should generate the next turn.
            formatted_prompt_key = tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True
            )
            prompt_to_gold_completion_map[formatted_prompt_key] = gold_completion

print(f"Generated prompt_to_gold_completion_map with {len(prompt_to_gold_completion_map)} entries.")

['prompt', 'category', 'difficulty', 'quality', 'reward_model_score', 'conversation_tokens']


In [27]:
# 2. INSERTED: Format Reward Function
def reward_format(completions, **kwargs):
    rewards = []
    # Pattern to check for matching open and close tags
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"

    # Access the global tokenizer to decode if necessary
    global tokenizer

    for completion in completions:
        completion_str = ""
        # The TRL trainer should ideally pass List[str] for completions.
        # This code makes it robust against other types if they appear.
        if isinstance(completion, str):
            completion_str = completion
        elif isinstance(completion, dict): # Handle single dict messages explicitly
            if 'content' in completion and isinstance(completion['content'], str):
                completion_str = completion['content']
            else:
                completion_str = str(completion) # Fallback for unexpected dict structure
        elif isinstance(completion, list): # Specific handling for Python lists
            # Case 1: List of dictionaries (chat messages)
            if all(isinstance(item, dict) for item in completion):
                if hasattr(tokenizer, 'chat_template') and tokenizer.chat_template is not None:
                    completion_str = tokenizer.apply_chat_template(completion, tokenize=False)
                else:
                    completion_str = str(completion)
            # Case 2: List of integers (token IDs)
            elif all(isinstance(item, int) for item in completion):
                if tokenizer:
                    completion_str = tokenizer.decode(completion, skip_special_tokens=True)
                else:
                    completion_str = str(completion)
            # Case 3: Mixed list or list of other types (e.g., [1, {'a':'b'}] or [1.0, 2.0])
            else:
                completion_str = str(completion) # Fallback to string conversion for complex lists
        elif isinstance(completion, torch.Tensor): # Specific handling for torch.Tensor
            # Assume torch.Tensor contains token IDs if it's numeric
            if tokenizer:
                completion_str = tokenizer.decode(completion, skip_special_tokens=True)
            else:
                completion_str = str(completion)
        else:
            completion_str = str(completion) # Catch all other unexpected types

        clean_completion = completion_str.strip()
        if re.match(pattern, clean_completion, re.DOTALL):
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards

# 3. Ground Truth Reward Function
def ground_truth_check_reward(prompts, completions, **kwargs):
    rewards = []
    reward_fn_kwargs = kwargs.get('reward_fn_kwargs', {})
    prompt_to_gold_completion_map = reward_fn_kwargs.get('prompt_to_gold_completion_map', {})

    global tokenizer # Access the global tokenizer

    for i in range(len(prompts)):
        current_prompt_obj = prompts[i] # This will be a List[Dict]
        current_completion = completions[i] # This will be a string

        # Format the current_prompt_obj (List[Dict]) into a string to match the map keys
        formatted_prompt_for_lookup = ""
        if isinstance(current_prompt_obj, list) and all(isinstance(item, dict) for item in current_prompt_obj):
            if hasattr(tokenizer, 'chat_template') and tokenizer.chat_template is not None:
                formatted_prompt_for_lookup = tokenizer.apply_chat_template(
                    current_prompt_obj,
                    tokenize=False,
                    add_generation_prompt=True # Must match how keys were created in the map
                )
            else:
                formatted_prompt_for_lookup = str(current_prompt_obj) # Fallback
        else:
            formatted_prompt_for_lookup = str(current_prompt_obj) # Fallback for unexpected prompt type

        gold_completion = prompt_to_gold_completion_map.get(formatted_prompt_for_lookup, None)

        # The completion generated by the model might need normalization (e.g., stripping whitespace)
        clean_current_completion = current_completion.strip() if isinstance(current_completion, str) else str(current_completion)

        if gold_completion and clean_current_completion == gold_completion:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards

# 4. Combined Reward Function
def combined_reward_function(prompts, completions, **kwargs):
    # The tokenizer is not passed via kwargs by GRPOTrainer, so pass it directly
    # to reward_format if it needs it, or rely on global scope.
    # For ground_truth_check_reward, tokenizer isn't directly needed for decoding completions.
    format_rewards = reward_format(completions, **kwargs)
    truth_rewards = ground_truth_check_reward(prompts, completions, **kwargs)

    combined = [ (f_r + t_r) / 2.0 for f_r, t_r in zip(format_rewards, truth_rewards) ]
    return combined

In [ ]:
# # training config
# training_args = GRPOConfig(
#     output_dir="output",
#     num_train_epochs=3,
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=2,
#     logging_steps=10,
# )

In [17]:
# more detailed config (from the website)
training_args = GRPOConfig(
    output_dir="output",
    num_train_epochs=3,
    num_generations=4,  # Number of completions to generate for each prompt
    per_device_train_batch_size=4,  # We want to get all generations in one device batch
    # Optional but useful
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    logging_steps=10,
    use_vllm=False  # Speed up generation, need to 'pip install trl[vllm]'. Disabled due to CUDA library error.
)

In [19]:
# train_grpo.py
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer, # Use processing_class for the tokenizer
    reward_funcs=combined_reward_function,
    train_dataset=new_train_dataset,
    eval_dataset=eval_dataset,
    reward_kwargs={'reward_args': {'prompt_to_gold_completion_map': prompt_to_gold_completion_map}} # Pass the map here
)

In [28]:
trainer.train()

TypeError: unhashable type: 'list'

In [ ]:
trainer.evaluate()